# GX_06 — Customer Segments: Which Groups Should the Bank Treat Differently?

**The question this notebook answers:** *What kinds of customers does the bank have, which of them leave, and what should it offer each group?*

### What this notebook is about
The churn model from GX_04 gives every customer a risk score: it says **who** is likely to leave. It does not say **what kind of customer** that is, so it cannot say **which offer** to make. A 35-year-old with €120,000 on deposit and a 62-year-old with one product need different conversations, even at the same risk.

Segmentation answers that. We let Spark group customers who look alike, **without telling it who left** (unsupervised learning, MLlib `KMeans`). Only after the groups exist do we look at how many in each group left.

In this notebook we:
1. **Choose** the customer characteristics that describe a customer's relationship with the bank.
2. **Choose** the number of segments, with measures of fit and with business sense.
3. **Describe and name** each segment, and look inside the risky ones (country, age, activity).
4. **Check** that the segments hold for customers they were not built on (validation split).
5. **Combine** segments with the churn model: where does the contact list go, and where does the model miss leavers?
6. **Save** the segmentation model and every customer's segment, for the graph and streaming notebooks.
7. **Recommend** one action per segment.

### What this notebook uses and produces
| | |
|---|---|
| **Reads** | `gold_churn_features` (7,000 training customers build the segments; 1,499 validation customers check them) and the churn model saved by GX_04 (`/Volumes/workspace/churn/models/churn_final_model`) |
| **Writes** | Table `customer_segments` (all 10,000 customers: id, split, segment) and the segmentation model + its settings in the `models` volume |

### Three rules
- **Built on the training customers only**, like the model. Validation customers are only used to check.
- **`Exited` is never an input.** Segments must exist for a new customer whose future is unknown; the churn rate is only used to *describe* a segment afterwards.
- **Test customers get a segment, but their churn is never looked at here.** The test set stays reserved for GX_05.

## Step 0 — Preparing the notebook

**What we do:** load the shared helper code. The segmentation helpers live in `src/segments.py`, so the streaming notebook can later put a new customer into a segment with exactly the same code.

**Two settings, and why:**
- `SEEDS = 1…10`: KMeans starts from random points and can end in a different (worse) answer depending on the start. We try 10 starts and keep the best one.
- `K_RANGE = 2…8`: the numbers of segments we compare. More than 8 groups would be too many for a retention team to act on.
- `CHOSEN_K = 5`: the number of segments we keep. It is set here so every later cell uses the same value; Step 3 shows the evidence behind it.

In [0]:
# Make the shared helpers in src/ importable (this notebook lives in notebooks/).
import os
import sys

REPO_ROOT = os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

import json

import matplotlib.pyplot as plt
import pandas as pd
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F

from src.config import (
    CHURN_MODEL_PATH, CHURN_MODEL_SETTINGS, CUSTOMER_SEGMENTS, GOLD_CHURN,
    MODELS_VOLUME, SEGMENT_MODEL_PATH, SEGMENT_MODEL_SETTINGS,
)
from src.ingest import write_table
from src.ml import KEY_COL, LABEL_COL, SPLIT_COL
from src.segments import (
    CLUSTER_COL, OLDER_SEGMENT_NAME, SEGMENT_FEATURES, SEGMENT_ID_COL, SEGMENT_NAME_COL,
    SEGMENT_VECTOR_COL, add_segment_labels, build_segment_prep_stages, fit_best_kmeans, fixed_layout,
    segment_lookup, segment_profile,
)

SEEDS = list(range(1, 11))   # 10 starting points per k; the lowest-cost result is kept
K_RANGE = range(2, 9)        # numbers of segments tried: 2 to 8
CHOSEN_K = 5                 # the number of segments we use - the decision is explained in Step 3

# Same colours as GX_03, so the slides look like one deck
COLOR_MAIN = "#27ae60"
COLOR_DARK = "#0b5345"
COLOR_GREY = "#7f8c8d"

print("Spark version:", spark.version)
print("Segmentation inputs:", SEGMENT_FEATURES)

## Step 1 — The data, in a fixed layout

**What we do:** read the gold table, keep the 7,000 training customers (to build the segments) and the 1,499 validation customers (to check them). The training customers are sorted by `CustomerId` and kept in **one partition** (`fixed_layout`).

**Why the fixed layout:** Spark's KMeans picks its starting points from a random sample, and that sample depends on how the rows happen to be split into partitions — not only on the seed. So the same code, same seed and same customers can give different segments on another cluster. Sorting the rows and using one partition removes that: the same input always gives the same segments. For 7,000 rows one partition is no problem; at millions of rows we would keep the partitions and rely on the saved model instead (Step 9).

In [0]:
gold = spark.table(GOLD_CHURN)

# Training customers in a fixed order and one partition (see src/segments.fixed_layout):
# this is what makes KMeans give the same segments on every run and every cluster.
train = fixed_layout(gold.filter(F.col(SPLIT_COL) == "train")).cache()
val = gold.filter(F.col(SPLIT_COL) == "val").cache()

for name, df in (("train", train), ("val", val)):
    row = df.agg(F.count("*").alias("n"), F.sum(LABEL_COL).alias("leavers")).first()
    print(f"{name:5s}: {row['n']:,} customers, {row['leavers']:,} leavers ({row['leavers'] / row['n']:.1%})")

**Result:** **7,000 training customers, 1,426 leavers (20.4%)**, and **1,499 validation customers, 305 leavers (20.3%)** — the split from GX_02, unchanged.

**Showing the problem:** the next cell fits the same KMeans (7 segments, seed 42) on the same training customers spread over 1, 2, 4 and 8 partitions — first in whatever order the data comes, then in the fixed layout. For each run it prints the cost (lower = better segments) and the churn rate of the riskiest segment.

In [0]:
# The same KMeans (7 segments, seed 42) on the same 7,000 customers, only spread over a
# different number of partitions - first as they come, then in the fixed layout.
demo_rows = []
unordered = gold.filter(F.col(SPLIT_COL) == "train")
for layout in ("as the data comes", "fixed layout"):
    for parts in (1, 2, 4, 8):
        df = unordered.repartition(parts)
        if layout == "fixed layout":
            df = fixed_layout(df)
        prepared = Pipeline(stages=build_segment_prep_stages()).fit(df).transform(df)
        model, runs = fit_best_kmeans(prepared, k=7, seeds=[42])
        churn = (model.transform(prepared).groupBy(CLUSTER_COL)
                 .agg(F.round(F.avg(LABEL_COL) * 100, 1).alias("churn")).agg(F.max("churn")).first()[0])
        demo_rows.append({"layout": layout, "partitions": parts,
                          "cost": round(runs[0][1], 1), "riskiest_segment_churn_pct": churn})
display(pd.DataFrame(demo_rows))

**Result:** as the data comes, the answer changes with the partitioning: the cost and the riskiest segment differ between runs, although nothing but the layout changed. In the fixed layout all four runs print the same line (cost **9,468.9**). That single start is also far from the best 7-segment answer Step 3 finds with 10 starts (cost 8,275.3). This is why every fit below uses `fixed_layout` **and** tries 10 starts.

## Step 2 — Which characteristics describe a customer?

KMeans groups customers by distance: two customers are "similar" if their values are close. So the choice of columns *is* the choice of what "similar" means. We use four:

| Column | Why it goes in |
|---|---|
| **Age** | Second-strongest link to leaving in GX_03 (Cramér's V 0.36); also a life stage, which changes the right offer. |
| **Balance** | Money the customer keeps with the bank (V 0.13). A saver and an empty account are different relationships. |
| **NumOfProducts** | Strongest link to leaving (V 0.39), and a lever: the bank can offer a product. |
| **IsActiveMember** | Activity (V 0.17), and a lever: the bank can re-engage a customer. |

**Left out on purpose:**
- **`Exited`** — see the rules above.
- **Tenure, credit score, salary, credit card** — GX_03 found no real link to leaving (all p > 0.05). In a distance they only add noise. Step 4 tests this instead of assuming it.
- **Country and gender** — they describe *who* the customer is, not *how* they use the bank, and gender is a sensitive attribute. Country is reported per segment in Step 6.
- **Engagement, ZeroBalance** — built from the columns above; using them too would count the same information twice.

**What we do first:** compare stayers and leavers on these four columns, as a reminder of why they matter.

In [0]:
# The same 4 columns, side by side for stayers and leavers (training customers only).
display(
    train.groupBy(LABEL_COL)
    .agg(
        F.count("*").alias("customers"),
        *[F.round(F.avg(c), 2).alias(f"avg_{c}") for c in SEGMENT_FEATURES],
    )
    .orderBy(LABEL_COL)
)

**Result:** leavers are **older** (44.9 vs 37.4 years on average), hold **more money** (€91,457 vs €73,337) and are **less often active** (35% vs 56%). The average number of products is almost the same (1.47 vs 1.54), but GX_03 showed why: products work in both directions (2 products 7.5% churn, 1 product 27.7%, 3–4 products 83–100%), and an average hides that.

**What we do next:** fit the preparation steps on the training customers. `VectorAssembler` puts the four columns into one vector; `StandardScaler` rescales each to mean 0 and standard deviation 1.

**Why scaling is essential here:** KMeans measures plain distances. Unscaled, a balance difference of €10,000 would count 1,000 times more than 10 years of age, and the segments would be about balance only. After scaling, one standard deviation of age weighs the same as one standard deviation of balance.

In [0]:
# Preparation (assembler + scaler) is fitted ONCE, on the training customers only.
prep_model = Pipeline(stages=build_segment_prep_stages()).fit(train)
train_prepared = prep_model.transform(train).cache()

scaler = prep_model.stages[-1]
display(pd.DataFrame({
    "column": SEGMENT_FEATURES,
    "mean (learned on train)": [round(float(v), 2) for v in scaler.mean],
    "std (learned on train)": [round(float(v), 2) for v in scaler.std],
}))

**Result:** the scaler learned, from the training customers only: age mean **38.94** (std 10.44), balance mean **€77,028** (std €62,463), products mean **1.53** (std 0.58), activity mean **0.52** (52% active).

## Step 3 — How many segments?

**How KMeans works:** we choose a number of segments, k. KMeans places k centre points, puts every customer in the segment of the nearest centre, moves each centre to the average of its customers, and repeats until nothing changes. The result depends on where the centres started, so for every k we run 10 starts (`fit_best_kmeans`) and keep the one with the lowest cost.

**What we measure for each k (2 to 8):**
- **Cost** — the total squared distance from each customer to the centre of their segment. It always falls when k grows; we look for the point where it stops falling much (the "elbow").
- **Silhouette** (MLlib `ClusteringEvaluator`) — how much closer each customer is to its own segment than to the next one. From −1 to 1; higher means clearer segments.
- **Starts reaching the best** — out of the 10 starts, how many found the best cost (within 0.1%). Low numbers mean the answer is easy to miss.
- **Smallest segment** — a segment of 2% of customers is too small for its own campaign.

In [0]:
# Silhouette: how clearly each customer sits in its own segment rather than the nearest
# other one (-1 to 1, higher = clearer). Spark computes it on the whole DataFrame.
evaluator = ClusteringEvaluator(featuresCol=SEGMENT_VECTOR_COL, predictionCol=CLUSTER_COL)
n_train = train.count()

search_rows, best_models = [], {}
for k in K_RANGE:
    model, runs = fit_best_kmeans(train_prepared, k, SEEDS)
    best_models[k] = model
    clustered = model.transform(train_prepared)
    best_cost = min(cost for _, cost in runs)
    smallest = clustered.groupBy(CLUSTER_COL).count().agg(F.min("count")).first()[0]
    search_rows.append({
        "k": k,
        "silhouette": round(evaluator.evaluate(clustered), 3),
        "cost": round(best_cost, 1),
        "starts_reaching_best": sum(cost <= best_cost * 1.001 for _, cost in runs),
        "smallest_segment_pct": round(smallest * 100 / n_train, 1),
    })

k_search = pd.DataFrame(search_rows)
k_search["cost_drop_pct"] = (-k_search["cost"].pct_change() * 100).round(1)
display(k_search)

**Result:**
- **Silhouette rises with every extra segment**, from **0.407** (k=2) to **0.587** (k=8).
- **Cost falls steadily** — by 21%, 17%, 17%, 16%, 14% and 8% for each extra segment. There is no sharp elbow.
- **Several starts are needed.** Only k=2 is found by almost every start (9 of 10). For k=3 to 8, only 1 to 3 of the 10 starts reach the best answer — one single run would usually return a worse segmentation.
- At **k=8** the smallest segment is **2.7%** of customers; up to k=7 every segment has at least 8.6%.

The two charts show the same thing. The dashed line marks `CHOSEN_K` (5); the reasons follow below.

In [0]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(k_search["k"], k_search["cost"], marker="o", color=COLOR_MAIN)
ax1.set_title("Cost: total distance to the segment centres (lower = tighter)")
ax1.set_xlabel("Number of segments (k)")
ax1.set_ylabel("Cost")

ax2.plot(k_search["k"], k_search["silhouette"], marker="o", color=COLOR_DARK)
ax2.set_title("Silhouette (higher = clearer separation)")
ax2.set_xlabel("Number of segments (k)")
ax2.set_ylabel("Silhouette")

for ax in (ax1, ax2):
    ax.axvline(CHOSEN_K, color=COLOR_GREY, linestyle="--", linewidth=1)
    ax.set_xticks(list(K_RANGE))
plt.tight_layout()
plt.show()

**Result:** both curves improve smoothly; neither picks a k on its own.

### Decision: 5 segments
1. **The measures alone cannot decide.** Silhouette keeps rising partly for a mechanical reason: activity is a yes/no column and products take only four values, so the data contains blocks of near-identical customers. Every extra segment can split off one more block, which always looks "clearer". That says little about whether the new segment is useful.
2. **Every one of the five segments is large enough** to act on: the smallest holds 10.1% of customers.
3. **Five segments can each be described in one line** for management (Step 5).
4. **A 6th segment brings no new kind of customer.** To check this, the next cell puts the k=5 and k=6 solutions side by side.

This is the `CHOSEN_K = 5` set in Step 0.

In [0]:
# The k=5 and k=6 solutions side by side: does a 6th segment bring a new kind of customer?
for k in (CHOSEN_K, CHOSEN_K + 1):
    print(f"k = {k}")
    display(
        segment_profile(best_models[k].transform(train_prepared), CLUSTER_COL)
        .select(CLUSTER_COL, "customers", "churn_pct", "avg_age", "avg_balance",
                "one_product_pct", "two_products_pct", "active_pct")
        .orderBy("churn_pct")
    )

**Result:** at k=6 the two active groups stay almost the same (1,213 and 1,771 customers vs 1,259 and 1,764) and the older customers keep their shape (659 instead of 708). The new 6th segment is **inactive customers with savings and 2 or more products** (769 customers), taken mostly from the two inactive groups. It leaves at **31.7%** — almost the same as the inactive savers with exactly 1 product (**32.4%**, 1,387 customers). Both would get the same treatment from the bank. A segment that changes no decision is not worth the extra complexity: **we keep 5.**

## Step 4 — Would more columns give better segments?

The project plan proposed seven clustering columns: our four plus **tenure, credit score and salary**. We left those three out because GX_03 found no link to leaving. Here we test that decision instead of assuming it.

**What we do:** build 5 segments from all seven columns, with exactly the same procedure (training customers, scaling, 10 starts), and look at two things:
- Do the extra columns **change the segments** — do they differ between segments?
- Are the segments **clearer or blurrier** (silhouette)?

In [0]:
# Plan's 7 columns = our 4 + the three that showed no link to leaving in GX_03.
PLAN_FEATURES = SEGMENT_FEATURES + ["Tenure", "CreditScore", "EstimatedSalary"]

plan_prep = Pipeline(stages=build_segment_prep_stages(PLAN_FEATURES)).fit(train)
plan_prepared = plan_prep.transform(train).cache()
plan_kmeans, _ = fit_best_kmeans(plan_prepared, CHOSEN_K, SEEDS)
plan_clustered = plan_kmeans.transform(plan_prepared)

display(
    plan_clustered.groupBy(CLUSTER_COL)
    .agg(
        F.count("*").alias("customers"),
        F.round(F.avg(LABEL_COL) * 100, 1).alias("churn_pct"),
        *[F.round(F.avg(c), 1).alias(f"avg_{c}") for c in PLAN_FEATURES],
    )
    .orderBy("churn_pct")
)
print("Silhouette with 7 columns:", round(evaluator.evaluate(plan_clustered), 3),
      "| with our 4 columns:", k_search.loc[k_search["k"] == CHOSEN_K, "silhouette"].iloc[0])
plan_prepared.unpersist()

**Result:** the three extra columns change nothing useful.
- **The five segments are almost the same as ours:** the same sizes within 7 customers and the same churn rates (10.4%, 12.2%, 21.3%, 27.8%, 37.3% vs our 10.5%, 12.3%, 21.2%, 27.8%, 37.4%).
- **The extra columns hardly differ between segments:** average tenure 4.8–5.2 years, credit score 646–656, salary €95,742–€101,665. No segment is defined by them.
- **Silhouette drops** to **0.256** with seven columns (0.492 with four). That is expected when three columns with no structure are added: customers spread out in three directions that separate nobody. Silhouettes measured on different columns are not strictly comparable, so this is only supporting evidence — the main evidence is the two points above.

**Decision:** keep the four columns. Same segments, simpler to explain.

## Step 5 — The five segments

**What we do:**
1. Combine the fitted preparation and the best k=5 KMeans into **one `PipelineModel`**, so a customer goes from raw columns to segment in one call (this is the object we save in Step 9).
2. Describe each segment with `segment_profile`: size, churn rate and the averages that characterise it.
3. Name the segments with a rule (`segment_name` in `src/segments.py`): average age 50 or more → **"Older customers"**; otherwise **active / inactive** (most customers active or not) plus **savers / little or no savings** (most customers with or without a balance).

**Why names come from a rule:** KMeans numbers its clusters 0–4 in no particular order. A rule on the profile gives every segment a name that matches its content. The `segment_id` (1–5) orders the segments by churn rate, lowest first.

In [0]:
# The final segmentation = the fitted preparation + the best k=5 KMeans, as ONE pipeline model.
segment_model = PipelineModel(stages=prep_model.stages + [best_models[CHOSEN_K]])

train_clustered = segment_model.transform(train)
lookup = segment_lookup(segment_profile(train_clustered, CLUSTER_COL).collect())
train_seg = add_segment_labels(train_clustered, lookup).cache()

train_profile = segment_profile(train_seg, SEGMENT_ID_COL, SEGMENT_NAME_COL)
display(train_profile)

**Result:** five clearly different groups.

| # | Segment | Customers | Churn | What defines it |
|---|---|---|---|---|
| 1 | **Active, little or no savings** | 1,259 (18.0%) | **10.5%** | All active; 88% hold no balance; 76% have 2 products; average age 36 |
| 2 | **Active savers** | 1,764 (25.2%) | **12.3%** | All active; average balance €122,619; 70% have 1 product; 36% German |
| 3 | **Inactive, little or no savings** | 1,462 (20.9%) | **21.2%** | None active; 80% hold no balance; 71% have 2 products |
| 4 | **Inactive savers** | 1,807 (25.8%) | **27.8%** | None active; average balance €124,981; 76% have 1 product; 39% German |
| 5 | **Older customers** | 708 (10.1%) | **37.4%** | Average age 60.7, 98% aged 50+; 83% active; balance mixed (31% none) |

**What this shows:**
- **Below 50, two levers sort customers: activity and savings.** Active customers leave at 10–12%, inactive ones at 21–28%. Among the inactive, those with money in the bank leave more.
- **Inactive savers combine the GX_03 warning signs** — inactive, one product, a balance, often German. With **502 leavers** they are the single biggest source of churn: 35% of all 1,426 leavers.
- **Could a simple GROUP BY have found this?** The four younger segments are close to a split by activity × savings, and a `GROUP BY` on those two columns would give something similar — but only once you know those are the two columns that matter, and where "older" starts. KMeans found both from the data without being told: which columns separate customers, and that from about age 47 up (no customer in this segment is younger) age matters more than activity or savings. The `GROUP BY` queries in Step 6 build on that result.
- **Customers with 3–4 products are not a segment.** They are the riskiest customers one by one (83–100% churn in GX_03), but only 3% of customers, spread over all segments (at most 8.6% of any segment). They are too few to form a group; the churn model catches them individually.

In [0]:
prof = train_profile.toPandas()   # 5 rows
overall = 100 * train_seg.agg(F.avg(LABEL_COL)).first()[0]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(prof[SEGMENT_NAME_COL], prof["churn_pct"], color=COLOR_MAIN)
for bar, churn, n, share, leavers in zip(bars, prof["churn_pct"], prof["customers"], prof["share_pct"], prof["leavers"]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f"{churn:.1f}%  ({n:,} customers = {share:.0f}%, {leavers} leavers)", va="center", fontsize=9)
ax.axvline(overall, color=COLOR_GREY, linestyle="--", linewidth=1, label=f"All customers ({overall:.1f}%)")
ax.set_xlim(0, prof["churn_pct"].max() * 1.9)
ax.set_xlabel("Churn rate (%)")
ax.set_title("Five customer segments: how many leave")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

**Sentence for the slide:** *Two segments — inactive savers and older customers — are 36% of the customers but 54% of the leavers (767 of 1,426).*

## Step 6 — Inside the segments: country and age

Segments were built without country and without splitting age finely. Two questions remain, both answered with **Spark SQL** on the segmented customers (`segmented` view):
1. **Is Germany's high churn (GX_03) explained by the kind of customers it has?** If German customers were simply mostly inactive savers, Germany would look normal *within* each segment.
2. **Who leaves among the older customers?** This segment is mostly active, yet 37% leave.

**What we do first:** churn rate per segment and country. `AVG(CASE WHEN … THEN Exited END)` averages `Exited` over one country only (other rows give NULL, which `AVG` skips), so one query gives all three countries side by side.

In [0]:
train_seg.createOrReplaceTempView("segmented")

display(spark.sql(f"""
    SELECT {SEGMENT_ID_COL}, {SEGMENT_NAME_COL},
           ROUND(AVG(CASE WHEN Geography = 'France'  THEN Exited END) * 100, 1) AS france_churn_pct,
           ROUND(AVG(CASE WHEN Geography = 'Germany' THEN Exited END) * 100, 1) AS germany_churn_pct,
           ROUND(AVG(CASE WHEN Geography = 'Spain'   THEN Exited END) * 100, 1) AS spain_churn_pct,
           SUM(CASE WHEN Geography = 'Germany' THEN 1 ELSE 0 END)               AS germany_customers
    FROM segmented
    GROUP BY {SEGMENT_ID_COL}, {SEGMENT_NAME_COL}
    ORDER BY {SEGMENT_ID_COL}
"""))

**Result:** **Germany leaves more in every segment.** It is not explained by the kind of customers it has.
- **Active savers:** Germany **17.6%** vs France 9.1% and Spain 9.9%.
- **Inactive savers:** Germany **37.6%** vs 20.3% and 24.3%.
- **Older customers:** Germany **51.2%** vs 32.0% and 31.5%.
- German customers are concentrated in the two saver segments (629 + 705 = 1,334 of 1,758, 76%) — Germany has no customers with an empty account (GX_03), so few German customers fall into the "little or no savings" segments.

So the segment says **what offer** fits; the country adds **how urgent** it is. A German inactive saver is the highest-risk combination below age 50.

**What we do next:** split the older customers by age band and activity.

In [0]:
display(spark.sql(f"""
    SELECT CASE WHEN Age >= 60 THEN '60+' WHEN Age >= 50 THEN '50-59' ELSE 'under 50' END AS age_band,
           CASE WHEN IsActiveMember = 1 THEN 'active' ELSE 'inactive' END              AS activity,
           COUNT(*)                    AS customers,
           SUM(Exited)                 AS leavers,
           ROUND(AVG(Exited) * 100, 1) AS churn_pct
    FROM segmented
    WHERE {SEGMENT_NAME_COL} = '{OLDER_SEGMENT_NAME}'
    GROUP BY 1, 2
    ORDER BY 1, 2
"""))

**Result:** among older customers, **activity decides almost everything.**
- **Inactive:** 43 of 45 aged 50–59 left (**95.6%**), and 69 of 78 aged 60+ (**88.5%**). Only 123 customers, but **112 of the segment's 265 leavers** (42%).
- **Active 50–59:** still **37.6%** (287 customers) — being active protects less at this age.
- **Active 60+:** only **12.8%** (281 customers) — the safest group in the segment.
- 17 active customers aged 47–49 also landed here; too few to matter.

For the bank: an older customer who **becomes inactive** is the clearest warning sign in the data.

## Step 7 — Do the segments hold for new customers?

**What we do:** put the 1,499 validation customers into the segments with the fitted model (`segment_model.transform`) — no refitting, exactly as a new customer would be assigned. Then compare segment sizes and churn rates with the training customers.

**Why:** segments are only useful if they describe customers in general, not just the 7,000 they were built on. If the validation churn rates were in a different order, the segments would be noise.

In [0]:
# Validation customers: put them into the segments learned on train (no refitting).
val_seg = add_segment_labels(segment_model.transform(val), lookup).cache()
val_profile = segment_profile(val_seg, SEGMENT_ID_COL, SEGMENT_NAME_COL)

display(
    train_profile.select(SEGMENT_ID_COL, SEGMENT_NAME_COL,
                         F.col("share_pct").alias("train_share_pct"),
                         F.col("churn_pct").alias("train_churn_pct"))
    .join(val_profile.select(SEGMENT_ID_COL,
                             F.col("customers").alias("val_customers"),
                             F.col("share_pct").alias("val_share_pct"),
                             F.col("churn_pct").alias("val_churn_pct")),
          on=SEGMENT_ID_COL)
    .orderBy(SEGMENT_ID_COL)
)

**Result:** the segments hold.
- **Sizes are almost identical:** every segment's share moves by at most 1.3 points (older customers 10.1% → 11.4%).
- **The risk order holds for the three riskiest segments:** older customers **34.5%**, inactive savers **27.2%**, inactive with little or no savings **18.1%**.
- **The two active segments come out equal on validation** (13.6% and 13.5%; 10.5% and 12.3% on training). On 264 and 370 customers, a 2-point gap is within chance. We treat them as **one low-risk tier**, not as a ranking.
- Every churn rate is within about 3 points of its training value.

## Step 8 — Segments and the churn model together

The model (GX_04) decides **who** to contact: every customer with predicted churn probability **≥ 0.35**. The segments say **what kind of customer** each one is. Crossing the two answers two questions:
1. **Where does the contact list go?** Which segments fill it?
2. **Where does the model miss leavers?** A segment whose leavers are rarely flagged needs another approach than a personal call.

**What we do:** load the final model and threshold that GX_04 saved (Step 12 there — run GX_04 first), score the validation customers, and summarise per segment.

**Why validation customers:** the model never trained on them, so their scores are honest. (The 0.35 threshold was chosen on these customers in GX_04; here we only describe *where* flagged customers sit, we do not re-judge the model.)

In [0]:
# The final churn model and its threshold, exactly as GX_04 saved them.
churn_model = PipelineModel.load(CHURN_MODEL_PATH)
with open(CHURN_MODEL_SETTINGS) as f:
    churn_settings = json.load(f)
THRESHOLD = churn_settings["threshold"]
print(f"Churn model: {churn_settings['model']} {churn_settings['layers']}, threshold {THRESHOLD}")

val_scored = (
    churn_model.transform(val_seg)
    .withColumn("p_churn", vector_to_array("probability")[1])
    .withColumn("contact", (F.col("p_churn") >= THRESHOLD).cast("int"))
    .cache()
)

total_contact = val_scored.agg(F.sum("contact")).first()[0]
segment_risk = (
    val_scored.groupBy(SEGMENT_ID_COL, SEGMENT_NAME_COL)
    .agg(
        F.count("*").alias("customers"),
        F.round(F.avg(LABEL_COL) * 100, 1).alias("actual_churn_pct"),
        F.round(F.avg("p_churn") * 100, 1).alias("avg_predicted_risk_pct"),
        F.sum("contact").alias("contacted"),
        F.round(F.avg("contact") * 100, 1).alias("contacted_pct_of_segment"),
        F.round(F.sum("contact") * 100 / total_contact, 1).alias("share_of_contact_list_pct"),
        F.sum(LABEL_COL).alias("leavers"),
        F.sum(F.col(LABEL_COL) * F.col("contact")).alias("leavers_caught"),
    )
    .withColumn("leavers_caught_pct", F.round(F.col("leavers_caught") * 100 / F.col("leavers"), 1))
    .orderBy(SEGMENT_ID_COL)
)
display(segment_risk)
print(f"Contact list: {total_contact} of {val_scored.count():,} validation customers (P(churn) >= {THRESHOLD})")

**How to read the table** (validation customers, per segment):
- `actual_churn_pct` vs `avg_predicted_risk_pct` — how many really left vs the model's average risk.
- `contacted`, `contacted_pct_of_segment` — customers above the threshold, and their share of the segment.
- `share_of_contact_list_pct` — the segment's share of the whole contact list.
- `leavers_caught` / `leavers` — leavers who were on the contact list; `leavers_caught_pct` is the model's recall *inside* the segment.

**Note:** the churn model is a neural network, and Spark's version of it can come out slightly different on another cluster even with the same seed (GX_04). The exact counts therefore belong to this run; the cell below prints them, and the text after it only states patterns that held in every fit we tried.

**What we do next:** chart, per segment, how many leavers the model caught and how many it missed.

In [0]:
risk = segment_risk.toPandas()   # 5 rows
missed = risk["leavers"] - risk["leavers_caught"]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(risk[SEGMENT_NAME_COL], risk["leavers_caught"], color=COLOR_DARK, label="Leavers on the contact list (caught)")
ax.barh(risk[SEGMENT_NAME_COL], missed, left=risk["leavers_caught"], color=COLOR_GREY, label="Leavers not on the list (missed)")
for y, (caught, total) in enumerate(zip(risk["leavers_caught"], risk["leavers"])):
    ax.text(total + 1, y, f"{caught}/{total} caught", va="center", fontsize=9)
ax.set_xlim(0, risk["leavers"].max() * 1.3)
ax.set_xlabel("Leavers among the validation customers")
ax.set_title("Where the model catches leavers, and where it misses them")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

# The numbers behind the result text, printed so they always match this run.
for r in risk.itertuples():
    print(f"{getattr(r, SEGMENT_NAME_COL):30s} flagged {r.contacted_pct_of_segment:5.1f}% of the segment, "
          f"{r.share_of_contact_list_pct:5.1f}% of the contact list, caught {r.leavers_caught}/{r.leavers} leavers "
          f"({r.leavers_caught_pct:.1f}%)")

**Result** (exact numbers in the printed lines above):
- **The contact list goes mostly to inactive savers and older customers** — close to 6 in 10 contacted customers. Inactive savers alone are the biggest part, over a third of the list.
- **In the inactive and older segments the model catches most leavers** — about two thirds or more of each segment's leavers are on the list.
- **Active savers are the model's blind spot.** It flags only about 1 in 10 of them and catches only about **1 in 4** of their leavers. Active customers with little or no savings fare better (about half caught). These customers look safe on every column the model sees, so their leavers are hard to spot one by one.

**What this means:** the risk score works where risk is visible (inactive, older). For active savers a personal call based on the score would miss most leavers; a broad, low-cost action for the whole segment fits better (Step 10).

## Step 9 — Saving the segmentation

**What we do:** save the segmentation `PipelineModel` (preparation + KMeans) to the `models` volume, next to the churn model, with a small settings file: the four input columns and which KMeans cluster is which named segment. Then load it back and check that every training customer lands in the same segment.

**Why:**
- **The graph notebook (GX_07)** compares customer communities with these segments.
- **The streaming notebook** can give every new customer a segment, so an alert says not only "high risk" but also "inactive saver" — and therefore which offer to make.
- **Reproducibility:** the saved model is the segmentation. Later notebooks load it instead of re-running KMeans.

In [0]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {MODELS_VOLUME}")
segment_model.write().overwrite().save(SEGMENT_MODEL_PATH)

# What a later notebook needs besides the model: the inputs and the cluster -> segment names.
with open(SEGMENT_MODEL_SETTINGS, "w") as f:
    json.dump({
        "features": SEGMENT_FEATURES,
        "k": CHOSEN_K,
        "segments": {str(c): {"segment_id": sid, "segment_name": name} for c, (sid, name) in lookup.items()},
        "trained_on": "gold_churn_features, split = 'train' (7,000 customers)",
    }, f, indent=2)

# Check: the reloaded model puts every training customer in the same segment as before.
reloaded = PipelineModel.load(SEGMENT_MODEL_PATH)
moved = (
    reloaded.transform(train).select(KEY_COL, F.col(CLUSTER_COL).alias("after"))
    .join(train_seg.select(KEY_COL, F.col(CLUSTER_COL).alias("before")), KEY_COL)
    .filter(F.col("after") != F.col("before"))
    .count()
)
print(f"Saved to {SEGMENT_MODEL_PATH}")
print(f"Training customers in a different segment after reloading: {moved}")
assert moved == 0, "The reloaded model assigns segments differently - do not use it."

**Result:** the model is saved and, reloaded, puts **0** training customers in a different segment.

**What we do next:** give all 10,000 customers their segment and save it as the table `customer_segments` (`CustomerId`, `split`, `segment_id`, `segment_name`), with GX_02's `write_table` (Delta, overwrite, so re-running rebuilds it).

**Why all 10,000:** other notebooks can join on `CustomerId` instead of re-applying the model. Assigning a segment to a test customer uses only their age, balance, products and activity — never their `Exited` value.

In [0]:
# Every customer (train, val and test) gets a segment. No churn label is used here:
# the test customers' Exited values are not read, only their segment is assigned.
customer_segments = (
    add_segment_labels(reloaded.transform(gold), lookup)
    .select(KEY_COL, SPLIT_COL, SEGMENT_ID_COL, SEGMENT_NAME_COL)
)
write_table(customer_segments, CUSTOMER_SEGMENTS)

saved = spark.table(CUSTOMER_SEGMENTS)
print(f"{CUSTOMER_SEGMENTS}: {saved.count():,} customers")
display(
    saved.groupBy(SEGMENT_ID_COL, SEGMENT_NAME_COL)
    .pivot(SPLIT_COL, ["train", "val", "test"])
    .count()
    .orderBy(SEGMENT_ID_COL)
)

**Result:** **10,000 customers** in `customer_segments`. Each split has a similar mix of segments (for example, older customers: 708 train, 171 validation, 149 test), as expected from the stratified, hash-based split of GX_02.

## Step 10 — One action per segment

Combining the segment profiles (Step 5), the country and age detail (Step 6), the check on new customers (Step 7) and the model view (Step 8):

| Priority | Segment | Share / churn | What we know | Recommended action |
|---|---|---|---|---|
| **1** | **Older customers** | 10% / 37% | 9 in 10 *inactive* older customers leave; active 60+ only 13% | Treat **"older customer becomes inactive"** as an alert: personal call from an advisor within weeks. A dedicated senior advisor for active 50–59 year-olds (still 38%). |
| **1** | **Inactive savers** | 26% / 28%, **35% of all leavers** | Money in the bank, no activity, mostly one product; Germany 38% | **Reactivation offer tied to their savings** (e.g. a savings or investment product as their *second* product). Start with German inactive savers. The model's contact list covers this segment well. |
| **2** | **Inactive, little or no savings** | 21% / 21% | Already 2 products, but inactive | **Low-cost digital re-engagement** (app and e-mail nudges); personal calls only for customers on the model's contact list. |
| **3** | **Active savers** | 25% / 12% | Low churn, but the model catches few of its leavers; Germany 18% vs 9–10% elsewhere | **Broad loyalty action** rather than calls: a second product for the 70% with only one, regular service check-ins. Watch German active savers. |
| **4** | **Active, little or no savings** | 18% / 10.5% | The healthiest group: active, mostly 2 products | **Keep as is**; no retention budget needed. |

**A caution for every offer:** a *second* product protects (7.5% churn), but GX_03 found that customers with 3–4 products leave at 83–100%. Offers should never push customers beyond two products.

## Summary

### What we did
- Grouped the 7,000 training customers into segments with MLlib **KMeans**, on four columns (age, balance, products, activity), scaled with a fitted **Pipeline**; the churn label was never an input.
- Chose **5 segments** by comparing k=2…8 on cost, **silhouette** (`ClusteringEvaluator`), stability over 10 starts and segment size — and by checking what a 6th segment would add.
- Tested the plan's 7-column alternative: the same five segments, and the three extra columns separate nobody.
- Profiled and named the segments, looked inside them with **Spark SQL** (country, age × activity), checked them on the **validation customers**, and crossed them with the **saved churn model**.
- Saved the segmentation `PipelineModel` and the `customer_segments` table for later notebooks.

### What we learned
1. **Five segments:** active with little or no savings (10.5% churn), active savers (12.3%), inactive with little or no savings (21.2%), inactive savers (27.8%) and older customers (37.4%).
2. **Inactive savers and older customers are 36% of customers but 54% of leavers.**
3. **Inactive older customers almost all leave** (88–96%); active customers over 60 rarely do (13%).
4. **Germany leaves more in every segment** — its problem is not the mix of customers it has.
5. **The segments hold on new customers** (validation), except that the two active segments are equally safe.
6. **The model catches most leavers where risk is visible, but few among active savers** — that segment needs a broad action, not calls.
7. **Spark's KMeans is not reproducible by default:** its result depends on how the data is partitioned. A fixed layout, 10 starts and a saved model make the segmentation the same on every run.

### Next: GX_07 — Do similar customers leave together? (GraphFrames)
GX_07 builds a **customer similarity graph**: every customer is a point, linked to the customers most like them (found with MLlib LSH, so Spark never has to compare all 50 million pairs). GraphFrames then finds communities and checks whether customers whose neighbours left are more likely to leave. The segments from this notebook are the reference: if graph communities only repeat the segments, the graph adds little; if they cut across them, they show a pattern the segments missed.